#BUILD BY: KARTIK KUMAR
#USN: 1CD22CS061

In [6]:
# ─── Backend server for MultiModal Video Summarization (Colab) ─── COMP Correct
# Paste & run this in a Colab cell (replace NOTEBOOK paths if needed)

# 0. (first time) installs + ffmpeg + pyngrok
!pip install -q flask flask-cors pyngrok
!apt-get -qq update && apt-get -qq install -y ffmpeg
# If required (only first time): uncomment and set your token
!ngrok authtoken 2wVOhcrhm9F6C9TUzmHet24C6Sf_6Mh2EmMTQ5jVUQx77tWkH

# 1. Imports & dirs
import os, subprocess, threading, traceback, time
from flask import Flask, request, jsonify, send_from_directory, send_file, make_response, abort
from flask_cors import CORS
from pyngrok import ngrok

os.makedirs('/content/input', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

# 2. Constants (adjust NOTEBOOK paths to your actual notebooks)
INPUT_PATH         = '/content/input/test.mp4'                          # incoming video
TIMEFILE_PATH      = '/content/input/time.txt'                          # timestamp file
PRESENTER_IMG_PATH = '/content/input/presenter_image'                   # will append ext when saved
# Raw outputs produced by your notebooks (different per mode)
RAW_SUMMARY_FULL      = '/content/summary_video.mp4'
RAW_SUMMARY_TIMESTAMP = '/content/timestamp_requested.mp4'
RAW_SUMMARY_PRESENTER = '/content/requested_presenter.mp4'

# Final re-encoded file and summary text (frontend expects these)
OUT_FIXED          = '/content/output/summary_fixed.mp4'                # final re-encoded
OUT_TEXT           = '/content/output/summary.txt'                      # final text summary

MODE_PATH          = '/content/input/mode.txt'

# Notebooks — change these if your notebook filenames differ
NOTEBOOK_FULL      = '/content/Full_Video_Summarization_final.ipynb'
NOTEBOOK_TIMESTAMP = '/content/Time_Stamp_Based_Final_.ipynb'
NOTEBOOK_PRESENTER = '/content/Face_Recognition_final.ipynb'

# Map mode -> raw output path (so backend knows which raw file to re-encode)
RAW_OUTPUT_BY_MODE = {
    'full': RAW_SUMMARY_FULL,
    'timestamp': RAW_SUMMARY_TIMESTAMP,
    'presenter': RAW_SUMMARY_PRESENTER
}

# 3. Flask app + CORS
app = Flask(__name__)
CORS(app)  # allow cross-origin from frontend

# 4. Re-encode function (makes MP4 web-friendly)
def reencode_video_for_web(input_path, output_path):
    try:
        print("🔁 Re-encoding", input_path, "->", output_path)
        subprocess.run([
            'ffmpeg', '-y', '-i', input_path,
            '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
            '-c:a', 'aac', '-b:a', '128k',
            '-movflags', '+faststart',
            output_path
        ], check=True)
        print("✅ Re-encoding complete.")
    except Exception as e:
        print("❌ FFmpeg re-encoding error:", e)

# 5. Helper: choose which notebook to run based on mode
def choose_notebook_for_mode(mode):
    m = (mode or '').lower()
    if m == 'timestamp':
        return NOTEBOOK_TIMESTAMP
    if m == 'presenter':
        return NOTEBOOK_PRESENTER
    # default to full
    return NOTEBOOK_FULL

# 6. Background notebook runner (selects raw output per mode, re-encodes appropriately)
def run_notebook():
    print("🔁 Starting notebook execution (background)...")
    try:
        mode = None
        if os.path.exists(MODE_PATH):
            try:
                with open(MODE_PATH, 'r') as fh:
                    mode = fh.read().strip()
                print("ℹ️ Notebook mode:", mode)
            except:
                pass

        # choose appropriate notebook
        notebook_path = choose_notebook_for_mode(mode)
        print("ℹ️ Executing notebook:", notebook_path)

        env = os.environ.copy()
        if mode:
            env['MODE'] = mode

        # Execute the notebook (nbconvert executes the notebook top-to-bottom)
        subprocess.run([
            'jupyter', 'nbconvert', '--to', 'notebook', '--execute', notebook_path,
            '--output', 'executed.ipynb',
            '--ExecutePreprocessor.timeout=3600'  # 60 min - increase if your runs take longer
        ], check=True, env=env)
        print("✅ Notebook execution finished.")

        # Determine which raw output to look for depending on mode
        chosen_mode = (mode or 'full').lower()
        raw_output = RAW_OUTPUT_BY_MODE.get(chosen_mode, RAW_SUMMARY_FULL)
        print("ℹ️ Expected raw output for mode", chosen_mode, "->", raw_output)

        # After notebook completes, re-encode the correct raw output to OUT_FIXED for browser compatibility
        if os.path.exists(raw_output):
            reencode_video_for_web(raw_output, OUT_FIXED)
        else:
            # maybe the notebook wrote OUT_FIXED directly — check that
            if os.path.exists(OUT_FIXED):
                print("ℹ️ OUT_FIXED already present (notebook may have written it):", OUT_FIXED)
            else:
                # Log the directory contents for debugging
                print("⚠️ Notebook finished but expected raw output not found:", raw_output)
                print("📁 /content directory listing:")
                try:
                    for entry in os.listdir('/content'):
                        print(" -", entry)
                except Exception as _:
                    pass

    except Exception:
        print("❌ Error during notebook execution:\n", traceback.format_exc())

# 7. /upload endpoint - receives form-data 'video' and optional 'mode' and optional 'timefile' or 'image'
@app.route('/upload', methods=['POST'])
def upload_video():
    try:
        f = request.files.get('video')
        mode = request.form.get('mode', '').strip().lower()

        if not f:
            return "No file received (field 'video')", 400

        # Save main video
        f.save(INPUT_PATH)
        print(f"📥 Saved uploaded video to {INPUT_PATH}")

        # Save mode string for notebook to consume
        if mode:
            with open(MODE_PATH, 'w') as fh:
                fh.write(mode)
            print("📌 Saved mode:", mode)
        else:
            if os.path.exists(MODE_PATH):
                os.remove(MODE_PATH)
                print("🧹 Removed previous mode file")

        # Save additional files depending on mode
        # - timestamp mode: expect 'timefile' field (text)
        # - presenter mode: expect 'image' field (png/jpg)
        if mode == 'timestamp':
            tf = request.files.get('timefile')
            if tf:
                # force name time.txt
                tf.save(TIMEFILE_PATH)
                print("📥 Saved timefile to", TIMEFILE_PATH)
            else:
                print("⚠️ timestamp mode but no 'timefile' provided")
        elif mode == 'presenter':
            img = request.files.get('image')
            if img:
                # preserve extension, save to PRESENTER_IMG_PATH + ext
                fname = img.filename or 'presenter_image'
                _, ext = os.path.splitext(fname)
                if not ext:
                    ext = '.png'
                save_path = PRESENTER_IMG_PATH + ext
                img.save(save_path)
                # also write a canonical path file so notebooks can find it
                try:
                    with open(PRESENTER_IMG_PATH + '.path', 'w') as ph:
                        ph.write(save_path)
                except Exception:
                    pass
                print("📥 Saved presenter image to", save_path)
            else:
                print("⚠️ presenter mode but no 'image' provided")

        # Remove old outputs to avoid confusion (remove all possible raw outputs + final outputs)
        for p in (RAW_SUMMARY_FULL, RAW_SUMMARY_TIMESTAMP, RAW_SUMMARY_PRESENTER, OUT_FIXED, OUT_TEXT):
            try:
                if os.path.exists(p):
                    os.remove(p)
                    print("🧹 Removed old file:", p)
            except Exception as e:
                print("❌ Error removing old file", p, e)

        # Start notebook execution in background thread (non-blocking)
        t = threading.Thread(target=run_notebook, daemon=True)
        t.start()

        return "Processing started", 200

    except Exception as e:
        print("❌ Upload error:", e)
        return str(e), 500

# 8. /check endpoint - frontend polls this
@app.route('/check', methods=['GET'])
def check_progress():
    # check the *re-encoded* fixed path for video readiness
    video_ready = os.path.exists(OUT_FIXED)
    text_ready  = os.path.exists(OUT_TEXT)
    print(f"[check] video_ready={video_ready} text_ready={text_ready} time={time.ctime()}")

    if video_ready:
        # read text if present (else empty string)
        summary = ''
        if text_ready:
            try:
                with open(OUT_TEXT, 'r', encoding='utf-8') as fh:
                    summary = fh.read()
            except Exception as e:
                print("❌ Error reading summary text:", e)
                summary = ''
        # Return JSON with video_url relative path (frontend will prepend BACKEND_URL)
        return jsonify(ready=True, summary=summary, video_url="/output/summary_fixed.mp4")
    return jsonify(ready=False)

# 9. Serve raw summary text at /summary.txt (plain text)
@app.route('/summary.txt')
def serve_summary_txt():
    if not os.path.exists(OUT_TEXT):
        return ("Not ready", 404)
    try:
        with open(OUT_TEXT, 'r', encoding='utf-8') as fh:
            txt = fh.read()
        resp = make_response(txt)
        resp.headers['Content-Type'] = 'text/plain; charset=utf-8'
        resp.headers['Access-Control-Allow-Origin'] = '*'
        resp.headers['Cache-Control'] = 'no-cache, no-store, must-revalidate'
        return resp
    except Exception as e:
        print("❌ serve_summary_txt error:", e)
        return ("Server error", 500)

# 10. Serve re-encoded summary directly at /summary.mp4 (optional)
@app.route('/summary.mp4')
def serve_summary_video():
    if not os.path.exists(OUT_FIXED):
        return ("Not ready", 404)
    try:
        resp = make_response(send_file(OUT_FIXED, mimetype='video/mp4'))
        resp.headers['Access-Control-Allow-Origin'] = '*'
        resp.headers['Cache-Control'] = 'no-cache, no-store, must-revalidate'
        return resp
    except Exception as e:
        print("❌ serve_summary_video error:", e)
        return ("Server error", 500)

# 11. Serve output folder generically (frontend uses this path)
@app.route('/output/<path:filename>')
def serve_output(filename):
    # safe serve from /content/output
    safe_name = os.path.basename(filename)
    full_path = os.path.join('/content/output', safe_name)
    if not os.path.exists(full_path):
        return ("Not ready", 404)
    return send_from_directory('/content/output', safe_name)

# 12. Start ngrok + Flask
if __name__ == '__main__':
    public_url = ngrok.connect(5000).public_url
    print("🚀 PUBLIC URL:", public_url)
    print("Set BACKEND_URL in your frontend to this value (no trailing slash).")
    app.run(port=5000, host='0.0.0.0')


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
🚀 PUBLIC URL: https://8ddbd158cd20.ngrok-free.app
Set BACKEND_URL in your frontend to this value (no trailing slash).
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 10:09:27] "POST /upload HTTP/1.1" 200 -


📥 Saved uploaded video to /content/input/test.mp4
📌 Saved mode: presenter
📥 Saved presenter image to /content/input/presenter_image.png
🔁 Starting notebook execution (background)...
ℹ️ Notebook mode: presenter
ℹ️ Executing notebook: /content/Face_Recognition_final.ipynb


INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 10:09:39] "HEAD /output/summary_fixed.mp4?nocache=1758103779008 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 10:09:49] "HEAD /output/summary_fixed.mp4?nocache=1758103789483 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 10:09:59] "HEAD /output/summary_fixed.mp4?nocache=1758103798999 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 10:10:09] "HEAD /output/summary_fixed.mp4?nocache=1758103808983 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 10:10:19] "HEAD /output/summary_fixed.mp4?nocache=1758103818961 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 10:10:29] "HEAD /output/summary_fixed.mp4?nocache=1758103829130 HTTP/1.1" 404 -


✅ Notebook execution finished.
ℹ️ Expected raw output for mode presenter -> /content/requested_presenter.mp4
🔁 Re-encoding /content/requested_presenter.mp4 -> /content/output/summary_fixed.mp4


INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 10:11:01] "HEAD /output/summary_fixed.mp4?nocache=1758103860959 HTTP/1.1" 200 -


✅ Re-encoding complete.


INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 10:11:43] "GET /output/summary_fixed.mp4?nocache=1758103861920 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 10:11:43] "GET /output/summary_fixed.mp4?nocache=1758103861920 HTTP/1.1" 206 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 10:11:49] "GET /output/summary_fixed.mp4?nocache=1758103861920 HTTP/1.1" 206 -
